In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np

# ============ 1. CHECK GPU ============
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Using: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:

DATA_DIR = "processed_data" 

train_dir = os.path.join(DATA_DIR, "train")
val_dir = os.path.join(DATA_DIR, "val")
test_dir = os.path.join(DATA_DIR, "test")

# ============ 3. DATA LOADERS ============
BATCH_SIZE = 32
IMG_SIZE = 224

# Simple transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [3]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=val_transform)

# Get class names
class_names = train_dataset.classes
num_classes = len(class_names)
print(f"\n📊 Found {num_classes} breeds:")
for i, name in enumerate(class_names):
    print(f"   {i+1}. {name}")

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n✅ Train: {len(train_dataset)} images")
print(f"✅ Val: {len(val_dataset)} images")
print(f"✅ Test: {len(test_dataset)} images")



📊 Found 50 breeds:
   1. Amritmahal
   2. Ayrshire
   3. Bargur
   4. Dangi
   5. Deoni
   6. Gir
   7. Hallikar
   8. Hariana
   9. Himachali Pahari
   10. Kangayam
   11. Kankrej
   12. Kenkatha
   13. Khariar
   14. Khillari
   15. Konkan Kapila
   16. Kosali
   17. Krishna_Valley
   18. Ladakhi
   19. Lakhimi
   20. Malnad_gidda
   21. Mewati
   22. Nari
   23. Nimari
   24. Ongole
   25. Poda Thirupu
   26. Pulikulam
   27. Punganur
   28. Purnea
   29. Rathi
   30. Red kandhari
   31. Red_Sindhi
   32. Sahiwal
   33. Shweta Kapila
   34. Tharparkar
   35. Umblachery
   36. Vechur
   37. bachaur
   38. badri
   39. bhelai
   40. dagri
   41. gangatari
   42. gaolao
   43. ghumsari
   44. kherigarh
   45. malvi
   46. motu
   47. nagori
   48. ponwar
   49. siri
   50. thutho

✅ Train: 5891 images
✅ Val: 1266 images
✅ Test: 1292 images


In [23]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import torch

# ============ 1. BUILD MODEL - CORRECTLY ============
print("\n🏗️ Building MobileNetV2...")

# Load pretrained model
model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

# ✅ FIX 1: MobileNetV2 uses classifier as Sequential
# Get the input features of the last layer
in_features = model.classifier[1].in_features

# Replace classifier properly
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, num_classes)
)

model = model.to(device)

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

# ============ 2. FIXED OPTIMIZER ============
# ✅ FIX 2: MobileNetV2 doesn't have layer1, layer2, etc. like ResNet
# It has different layer structure: features (0-17) and classifier

# Get layer groups for discriminative learning rates
feature_params = []
classifier_params = []

for name, param in model.named_parameters():
    if 'classifier' in name:
        classifier_params.append(param)
    else:
        feature_params.append(param)

# ✅ CORRECT OPTIMIZER FOR MOBILENETV2
optimizer = optim.AdamW([
    {'params': feature_params, 'lr': 1e-4},        # Feature layers
    {'params': classifier_params, 'lr': 1e-3}      # Classifier layers
], weight_decay=1e-4)

criterion = nn.CrossEntropyLoss()

print(f"✅ Model ready with {num_classes} output classes")
print(f"   Feature layers: {len(feature_params)} parameter groups")
print(f"   Classifier layers: {len(classifier_params)} parameter groups")

# ============ 3. SCHEDULER ============
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    patience=3, 
    factor=0.5
)

# ============ 4. TRAINING FUNCTIONS ============
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss/len(loader), 100.*correct/total

def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss/len(loader), 100.*correct/total

# ============ 5. TRAIN ============
EPOCHS = 15
best_val_acc = 0.0
patience = 5
patience_counter = 0

print("\n" + "="*50)
print("STARTING TRAINING WITH MOBILENETV2")
print("="*50)

train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-"*40)
    
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # Step scheduler
    scheduler.step(val_loss)
    
    print(f"\n📊 Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"   Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print(f"   LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model with early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'class_names': class_names,
        }, 'best_model.pth')
        patience_counter = 0
        print(f"   💾 Saved best model (Val Acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"   ⏹️ Early stopping triggered!")
            break

print("\n" + "="*50)
print(f"🏆 Best Validation Accuracy: {best_val_acc:.2f}%")

# ============ 6. TEST ============
print("\n📊 Testing best model...")
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_acc = validate(model, test_loader, criterion)
print(f"🎯 Test Accuracy: {test_acc:.2f}%")

# ============ 7. SAVE LABELS ============
with open('labels.txt', 'w') as f:
    for name in class_names:
        f.write(f"{name}\n")
print("✅ Saved labels.txt")


🏗️ Building MobileNetV2...
✅ Model ready with 50 output classes
   Feature layers: 156 parameter groups
   Classifier layers: 4 parameter groups

STARTING TRAINING WITH MOBILENETV2

Epoch 1/15
----------------------------------------


Validating: 100%|█████████████████████████████████████████████████████████████| 40/40 [00:07<00:00,  5.19it/s]



📊 Results:
   Train Loss: 2.5609 | Train Acc: 29.33%
   Val Loss: 1.6214 | Val Acc: 49.61%
   LR: 0.000100
   💾 Saved best model (Val Acc: 49.61%)

Epoch 2/15
----------------------------------------


Validating: 100%|█████████████████████████████████████████████████████████████| 40/40 [00:07<00:00,  5.16it/s]



📊 Results:
   Train Loss: 1.5464 | Train Acc: 52.08%
   Val Loss: 1.3339 | Val Acc: 58.69%
   LR: 0.000100
   💾 Saved best model (Val Acc: 58.69%)

Epoch 3/15
----------------------------------------


...


📊 Results:
   Train Loss: 0.3254 | Train Acc: 89.39%
   Val Loss: 1.3240 | Val Acc: 66.82%
   LR: 0.000025

Epoch 15/15
----------------------------------------


Validating: 100%|█████████████████████████████████████████████████████████████| 40/40 [00:07<00:00,  5.19it/s]



📊 Results:
   Train Loss: 0.2959 | Train Acc: 90.07%
   Val Loss: 1.3803 | Val Acc: 65.80%
   LR: 0.000025

🏆 Best Validation Accuracy: 67.93%

📊 Testing best model...


Validating: 100%|█████████████████████████████████████████████████████████████| 41/41 [00:07<00:00,  5.53it/s]

🎯 Test Accuracy: 63.78%
✅ Saved labels.txt


In [33]:
# ============================================================
# SIMPLIFIED EXPORT - WORKS WITH ANY MOBILENETV2 CHECKPOINT
# ============================================================

import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
import numpy as np

# Load checkpoint
checkpoint = torch.load('best_model.pth', map_location='cpu')

# Create model
model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

# Get num_classes from checkpoint
state_dict = checkpoint['model_state_dict']
num_classes = state_dict['classifier.4.weight'].shape[0]  # Get from FC layer

# Replace classifier
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, num_classes)
)

# Remove 'backbone.' prefix if it exists
if any(key.startswith('backbone.') for key in state_dict.keys()):
    print("⚠️ Removing 'backbone.' prefix from state dict...")
    new_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith('backbone.'):
            new_key = key[9:]  # Remove 'backbone.'
        else:
            new_key = key
        new_state_dict[new_key] = value
    state_dict = new_state_dict

# Load state dict
model.load_state_dict(state_dict)
model.eval()
model.to('cpu')

print(f"✅ Model loaded with {num_classes} classes")

# Export to ONNX
dummy_input = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model,
    dummy_input,
    'model.onnx',
    export_params=True,
    opset_version=18,
    input_names=['input'],
    output_names=['output']
)
print("✅ Exported to model.onnx")

# Save labels
class_names = checkpoint.get('class_names', [f'Breed_{i}' for i in range(num_classes)])
with open('labels.txt', 'w') as f:
    for name in class_names:
        f.write(f"{name}\n")
print(f"✅ Saved labels.txt with {len(class_names)} classes")

✅ Model loaded with 50 classes
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/maddalajashwanth/.local/share/uv/python/cpython-3.11.16-linux-x86_64-gnu/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ Exported to model.onnx
✅ Saved labels.txt with 50 classes


In [38]:
# convert_to_tflite.py
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
import tensorflow as tf
import numpy as np
import onnx
from onnx2tf import convert

# Load your trained model
checkpoint = torch.load('best_model.pth', map_location='cpu')

# Create model
model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
num_classes = checkpoint['model_state_dict']['classifier.4.weight'].shape[0]

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, num_classes)
)

# Load weights
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Export to ONNX first
dummy_input = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model,
    dummy_input,
    'model.onnx',
    export_params=True,
    opset_version=11,
    input_names=['input'],
    output_names=['output']
)

# Convert ONNX to TFLite
convert(
    input_onnx_file_path='model.onnx',
    output_folder_path='./',
    output_signaturedefs=True,
    replace_to_float16=False
)

print("✅ Model converted to TFLite")

# Save labels
class_names = checkpoint.get('class_names', [f'Breed_{i}' for i in range(num_classes)])
with open('labels.txt', 'w') as f:
    for name in class_names:
        f.write(f"{name}\n")

print("✅ Labels saved")

ModuleNotFoundError: No module named 'onnx2tf'

In [39]:
!uv pip install onnx2tf

Using Python 3.11.16 environment at: /home/maddalajashwanth/Documents/DL/.venv
Resolved 79 packages in 1.96s                                        
⠦ Preparing packages... (0/1)                                                   ^C
